In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
from sklearn.model_selection import StratifiedKFold

print(f'LightGBM version: {lgb.__version__}')
print('Imports OK')

In [ ]:
SEED     = 42
N_TRIALS = 20
np.random.seed(SEED)

BASE_DIR = Path('../../')
DATA_DIR = BASE_DIR / 'data/processed'

matplotlib.rcParams.update({'figure.figsize': (9, 6), 'font.size': 12,
                            'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
df      = pd.read_parquet(DATA_DIR / 'features_train.parquet')
df_test = pd.read_parquet(DATA_DIR / 'features_test.parquet')

feature_cols = [c for c in df.columns if c not in ('SK_ID_CURR', 'TARGET')]

X           = df[feature_cols].values
y           = df['TARGET'].values
X_test      = df_test[feature_cols].values
sk_ids_test = df_test['SK_ID_CURR'].values

scale_pos_weight = (y == 0).sum() / (y == 1).sum()

print(f'Train: {X.shape}  | Default rate: {y.mean():.2%}')
print(f'Test : {X_test.shape}')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5, label='Model'):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    auc  = roc_auc_score(y_true, y_prob)
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(Model=label, AUC=round(auc,4),
                N=len(y_true), P=int(y_pred.sum()),
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                Recall=round(rec,4), Precision=round(prec,4), F1=round(f1,4))

def plot_roc_curve(y_true, y_prob, label, color, ax, title='ROC Curve'):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC = {auc:.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(title); ax.legend()
    return ax

print('Funciones definidas OK')

---
## Optuna — Búsqueda de Hiperparámetros

Usa `lgb.cv()` con `early_stopping_rounds=50` (equivalente a `xgb.cv`).
El objetivo penaliza el overfitting: `CV_AUC - 0.5 * max(0, Train_AUC - CV_AUC)`.

In [ ]:
dtrain = lgb.Dataset(X, label=y, feature_name=feature_cols, free_raw_data=False)

PARAMS_FIXED = {
    'objective'        : 'binary',
    'metric'           : 'auc',
    'scale_pos_weight' : scale_pos_weight,
    'verbosity'        : -1,
    'seed'             : SEED,
}

def objective(trial):
    params = {
        **PARAMS_FIXED,
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 300),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 3, 9),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }

    cv_result = lgb.cv(
        params,
        dtrain,
        num_boost_round=1000,
        nfold=5,
        stratified=True,
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
        seed=SEED,
    )

    n_rounds  = len(cv_result['valid auc-mean'])
    cv_auc    = cv_result['valid auc-mean'][-1]

    # Train AUC con los mismos params pero en todo el train
    booster = lgb.train(params, dtrain, num_boost_round=n_rounds,
                        callbacks=[lgb.log_evaluation(-1)])
    train_auc = roc_auc_score(y, booster.predict(X))

    trial.set_user_attr('n_rounds', n_rounds)
    trial.set_user_attr('train_auc', train_auc)

    gap = max(0, train_auc - cv_auc)
    return cv_auc - 0.5 * gap

print(f'Lanzando Optuna — {N_TRIALS} trials...')
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_trial
print(f'\nMejores hiperparámetros:')
for k, v in best.params.items():
    print(f'  {k:<22}: {v}')
print(f'  n_rounds (óptimo) : {best.user_attrs["n_rounds"]}')
print(f'\nObjetivo (CV AUC penalizado): {best.value:.4f}')
print(f'CV AUC  : {study.best_trial.user_attrs["train_auc"]:.4f}  (train, referencia)')

In [ ]:
# Refit en todo el train con los mejores hiperparámetros
best_params = {**PARAMS_FIXED, **study.best_trial.params}
best_rounds = study.best_trial.user_attrs['n_rounds']

model = lgb.train(
    best_params,
    dtrain,
    num_boost_round=best_rounds,
    callbacks=[lgb.log_evaluation(100)],
)

y_prob = model.predict(X)

# CV AUC final (5-fold stratified con modelo refiteado)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_aucs = []
for tr_i, val_i in skf.split(X, y):
    ds_tr  = lgb.Dataset(X[tr_i], label=y[tr_i])
    m_fold = lgb.train(best_params, ds_tr, num_boost_round=best_rounds,
                       callbacks=[lgb.log_evaluation(-1)])
    cv_aucs.append(roc_auc_score(y[val_i], m_fold.predict(X[val_i])))
cv_auc_final = np.mean(cv_aucs)
cv_auc_std   = np.std(cv_aucs)

metrics = compute_metrics(y, y_prob, threshold=0.5, label='LightGBM')
metrics['CV_AUC']     = round(cv_auc_final, 4)
metrics['CV_AUC_std'] = round(cv_auc_std, 4)
metrics['n_rounds']   = best_rounds

print('\nLIGHTGBM — MÉTRICAS:')
for k, v in metrics.items():
    if k != 'Model':
        print(f'  {k:<14}: {v}')

---
## Visualizaciones

In [ ]:
# Optuna: historial de trials
trial_df = study.trials_dataframe()[['number','value']].rename(columns={'value':'Objetivo'})

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(trial_df['number'], trial_df['Objetivo'], 'o-', color='#2ecc71')
ax.axhline(study.best_value, color='#e74c3c', ls='--',
           label=f'Best = {study.best_value:.4f} (trial {study.best_trial.number})')
ax.set_xlabel('Trial'); ax.set_ylabel('CV AUC penalizado')
ax.set_title('Optuna — LightGBM'); ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fig, ax = plt.subplots(figsize=(8, 6))
plot_roc_curve(y, y_prob, label='LightGBM', color='#2ecc71', ax=ax,
               title='ROC Curve — LightGBM')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (Gain)
importance = model.feature_importance(importance_type='gain')
feat_imp_df = pd.DataFrame({'feature': feature_cols, 'importance': importance})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(feat_imp_df['feature'], feat_imp_df['importance'], color='#2ecc71', alpha=0.8)
ax.set_xlabel('Importance (Gain)'); ax.set_title('Feature Importance — LightGBM (Top 20)')
plt.tight_layout()
plt.show()

In [ ]:
# Score distribution TARGET=0 vs TARGET=1
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(y_prob[y == 0], bins=60, alpha=0.6, color='#3498db', label='TARGET=0 (paga)', density=True)
ax.hist(y_prob[y == 1], bins=60, alpha=0.6, color='#e74c3c', label='TARGET=1 (default)', density=True)
ax.set_xlabel('Probabilidad predicha'); ax.set_ylabel('Densidad')
ax.set_title('Distribución de scores — LightGBM')
ax.legend()
plt.tight_layout()
plt.show()

---
## Predicciones Test

In [ ]:
y_pred_test = model.predict(X_test)
submission = pd.DataFrame({'SK_ID_CURR': sk_ids_test, 'TARGET': y_pred_test})
_sub_path = DATA_DIR / 'submission_11_lgbm.csv'
submission.to_csv(_sub_path, index=False)
print(f'Submission guardada: {_sub_path.name}  ({len(submission):,} filas)')
print(f'TARGET stats: mean={y_pred_test.mean():.4f}  min={y_pred_test.min():.4f}  max={y_pred_test.max():.4f}')

---
## Resumen Final

In [ ]:
print('=' * 65)
print('LIGHTGBM — RESUMEN')
print('=' * 65)
print(f'  num_leaves       : {best_params["num_leaves"]}')
print(f'  learning_rate    : {best_params["learning_rate"]:.5f}')
print(f'  max_depth        : {best_params["max_depth"]}')
print(f'  n_rounds         : {best_rounds}')
print(f'  Train AUC        : {metrics["AUC"]}')
print(f'  CV AUC (5-fold)  : {metrics["CV_AUC"]} ± {metrics["CV_AUC_std"]}')
print(f'  Recall           : {metrics["Recall"]}')
print(f'  Precision        : {metrics["Precision"]}')
print(f'  F1               : {metrics["F1"]}')
print('=' * 65)

---
## Kaggle Submission — AUC Test (Public Leaderboard)

Envía el CSV a `home-credit-default-risk` y recupera el AUC del Public LB (~30% del test).

> **Límite**: 5 submissions/día.

In [ ]:
from kaggle import KaggleApi
import time

COMPETITION = 'home-credit-default-risk'
_msg = f'11_lgbm | train={metrics["AUC"]:.4f} | cv={metrics["CV_AUC"]:.4f} | rounds={best_rounds}'

def _as_str(x):
    return '' if x is None else str(x)

def _get_score(api, comp, file_name, message, poll=10, timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        subs = api.competition_submissions(comp)
        matched = next(
            (s for s in subs
             if _as_str(getattr(s, 'file_name', None)) == file_name
             and _as_str(getattr(s, 'description', None)) == message),
            subs[0] if subs else None
        )
        if matched:
            pub     = _as_str(getattr(matched, 'public_score',  None))
            status  = _as_str(getattr(matched, 'status',        None))
            elapsed = int(time.time() - start)
            print(f'  [{elapsed:>3}s] status={status!r}  public_score={pub!r}')
            if pub and pub.lower() not in ('', 'none', 'null', '-'):
                priv = _as_str(getattr(matched, 'private_score', None))
                return float(pub), (float(priv) if priv and priv.lower() not in ('','none','null','-') else None)
        time.sleep(poll)
    return None, None

_api = KaggleApi()
_api.authenticate()

print(f'Enviando: {_msg}')
_api.competition_submit(file_name=str(_sub_path), message=_msg, competition=COMPETITION)
print('Esperando scoring (poll 10 s / máx 5 min)...')

public_auc, private_auc = _get_score(_api, COMPETITION, _sub_path.name, _msg)
print(f'\nAUC test  Public  LB : {public_auc}')
print(f'AUC test  Private LB : {private_auc}')